# 0. Import libraries

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

from xclim.sdba.adjustment import QuantileDeltaMapping
from xclim.sdba import Grouper

# 1. Read in historical observations at the selected station

In [ ]:
obs_df = pd.read_csv('historical_observations_at_stationX.csv')

obs_df['time'] = pd.to_datetime(obs_df['time'])
obs_df = obs_df[(obs_df['time'].dt.year >= 1981) & (obs_df['time'].dt.year <= 2014)]
obs_df = obs_df[['time', 't2m_obs']]

obs_ds = obs_df.set_index(['time']).to_xarray()
obs_ds = obs_ds.assign_attrs(units='degF').convert_calendar("noleap")

# 2. Read in WRF simulation data for the nearest gridcell

In [ ]:
def read_gridded_WRF (station_name):

    year_start = 1981
    year_end = 2100
    
    wrf_df = pd.read_csv(f'hourlyTEMP_SSP3-7.0_{station_name}_closestgridcell_3km.csv', index_col=[0], header=[0, 1, 2])
    wrf_df.index = pd.to_datetime(wrf_df.index)
    wrf_df = wrf_df.stack(future_stack=True).stack(future_stack=True)
    wrf_df = wrf_df.reset_index()
    wrf_df = wrf_df[(wrf_df['time'].dt.year >= year_start) & (wrf_df['time'].dt.year <= year_end)]
    wrf_df = wrf_df.rename(columns={station_name: 't2m_gridded'}) 
    wrf_df = wrf_df[['scenario', 'simulation', 'time', 't2m_gridded']]
    wrf_df = wrf_df.set_index(['scenario', 'simulation', 'time'])
    wrf_df = wrf_df.sort_index(level=['scenario', 'simulation', 'time'])
    wrf_df = wrf_df.astype('float32')
    wrf_df
    
    wrf_ds = wrf_df.reset_index()
    wrf_ds = wrf_ds.set_index(['scenario', 'simulation', 'time']).to_xarray()
    wrf_ds = wrf_ds.assign_attrs(units='degF').convert_calendar("noleap")

    return wrf_df, wrf_ds

# 3. Localize based on quantile delta mapping (QDM) method

In [ ]:
def run_QDM(obs_ds, wrf_ds, scenario, simulation):

    ref = obs_ds['t2m_obs'].assign_attrs(units='degF')
    sim = wrf_ds.sel(scenario=scenario, simulation=simulation)['t2m_gridded'].assign_attrs(units='degF')
    QDM = QuantileDeltaMapping.train(
            #ref, sim.sel(time=slice(ref.time.min(), ref.time.max())),
            ref, sim.sel(time=np.intersect1d(sim.time.values, ref.time.values)),
            nquantiles = 20, 
            group = Grouper('time.dayofyear', window=90), 
            kind = "+")
    
    sim_adj = QDM.adjust(sim)
    sim_adj.name = 't2m_localized'
    
    return QDM, sim_adj

# 4. Loop through simulations (GCM) and saved localized results in a csv file

In [ ]:
allsim_adj = None
wrf_df, wrf_ds = read_gridded_WRF (station_name)

for simulation in wrf_ds['simulation'].values:
    scenario = 'Historical + SSP 3-7.0 -- Business as Usual'
    QDM, sim_adj = run_QDM(scenario, simulation)
    sim_adj = sim_adj.expand_dims({'scenario': [scenario], 'simulation': [simulation]})
    allsim_adj = sim_adj if allsim_adj is None else xr.merge([allsim_adj, sim_adj])

allsim_adj['time'] = allsim_adj.indexes['time'].to_datetimeindex(unsafe=True)
allsim_adj = allsim_adj.to_dataframe().reset_index()
allsim_adj = allsim_adj.set_index(['scenario', 'simulation', 'time'])
allsim_adj = allsim_adj.sort_index(level=['scenario', 'simulation', 'time'])
allsim_adj = allsim_adj.astype('float32')

df = wrf_df.merge(allsim_adj, left_index=True, right_index=True, how='inner')
df = wrf_df[['t2m_localized']].rename(columns={'t2m_localized': station_name})
df = df.unstack('scenario').unstack('simulation')
df = df.sort_values(by=['time'])
df = df.sort_index(axis=1)
df.to_csv('hourlyTemp_SSP3-7.0_' + station_name +'_3km_extended_WRF_results.csv')
df